# Prediction analysis

This is to analyse the predicted ages.

In [ ]:
import os
# Temporary workaround: allow duplicate OpenMP runtimes so the kernel can start.
# NOTE: This is unsafe and can mask real issues. See safer fixes below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

# Load the predicted ages
results_df = pd.read_csv('../../Compare_Model/QSM+MD+Thickness_subject_level_mean_raw_and_corrected_pred_on_age.csv')

# Load metadata
METADATA_PATH = "../../../AD_DECODE_data3.xlsx"
metadata_df = pd.read_excel(METADATA_PATH, sheet_name='AD_DECODE_data2')
# Merge results with metadata by 'subject_id' in result_df and 'MRI_Exam' in metadata_df
# subject_id is like 02110
# MRI_Exam is like 2110
# Convert 'MRI_Exam' to match 'subject_id' format
metadata_df['subject_id'] = metadata_df['MRI_Exam'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
# Ensure both subject_id columns are strings for merging
results_df['subject_id'] = results_df['subject_id'].apply(lambda x: f"{int(x):05d}")
# Merge the dataframes
merged_df = pd.merge(results_df, metadata_df, on='subject_id', how='left')

## cBAG vs. cognitive score


In [ ]:
# Plot the cBAG vs column from 'MOCA_TOTAL' to 'Delayed_paraphrase'
import seaborn as sns

cognitive_start = merged_df.columns.get_loc('MOCA_TOTAL')
cognitive_end = merged_df.columns.get_loc('Delayed_paraphrase') + 1
cognitive_columns = merged_df.columns[cognitive_start:cognitive_end]

# Convert all cognitive columns to numeric, coercing errors to NaN
for col in cognitive_columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

R2 = []
p_values = []

for column in cognitive_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/cBAG_vs_{column}.png')

# Print the top 10 cognitive columns with the highest R2 values
cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df = cognitive_r2_df.sort_values(by='R2', ascending=False)
# Save the R2 and p-values to a CSV file
cognitive_r2_df.to_csv('biological_metrics/cBAG_vs_cognition_r2_p_values.csv', index=False)
# Print the top 10 cognitive columns with highest R² values
top_10_cognitive = cognitive_r2_df.head(10)
print("Top 10 cognitive columns with highest R² values:")
print(top_10_cognitive)
# Print the top 10 cognitive columns with lowest p-values
top_10_cognitive_p = cognitive_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 cognitive columns with lowest p-values:")
print(top_10_cognitive_p)


In [ ]:
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv('biological_metrics/cBAG_vs_cognition_r2_FDR_corrected_p_values.csv', index=False)


## BAG vs. cognitive score

In [ ]:
# Plot the BAG vs column from 'MOCA_TOTAL' to 'Delayed_paraphrase'
import seaborn as sns

cognitive_start = merged_df.columns.get_loc('MOCA_TOTAL')
cognitive_end = merged_df.columns.get_loc('Delayed_paraphrase') + 1
cognitive_columns = merged_df.columns[cognitive_start:cognitive_end]

# Convert all cognitive columns to numeric, coercing errors to NaN
for col in cognitive_columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

R2 = []
p_values = []

for column in cognitive_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/BAG_vs_{column}.png')

# Print the top 10 cognitive columns with the highest R2 values
cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df = cognitive_r2_df.sort_values(by='R2', ascending=False)
cognitive_r2_df.to_csv('biological_metrics/BAG_vs_cognition_r2_p_values.csv', index=False)
top_10_cognitive = cognitive_r2_df.head(10)
print("Top 10 cognitive columns with highest R² values:")
print(top_10_cognitive)
# Print the top 10 cognitive columns with lowest p-values
top_10_cognitive_p = cognitive_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 cognitive columns with lowest p-values:")
print(top_10_cognitive_p)


In [ ]:
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv('biological_metrics/cBAG_vs_cognition_r2_FDR_corrected_p_values.csv', index=False)


## cBAG vs. biological metrics

In [ ]:
import seaborn as sns
from scipy.stats import linregress

# Plot cBAG vs biological metrics with regression line, R2, and p-value

biological_start = merged_df.columns.get_loc('Systolic')
biological_end = merged_df.columns.get_loc('BMI') + 1
biological_columns = merged_df.columns[biological_start:biological_end]

R2 = []
p_values = []

for column in biological_columns:
    x = merged_df[column]
    y = merged_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/cBAG_vs_{column}.png')

cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df.to_csv('biological_metrics/cBAG_vs_biological_metrics_r2_p_values.csv', index=False)
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv('biological_metrics/cBAG_vs_biological_metrics_FDR_corrected_p_values.csv', index=False)

## BAG vs. biological metrics

In [ ]:
import seaborn as sns
from scipy.stats import linregress

# Plot cBAG vs biological metrics with regression line, R2, and p-value

biological_start = merged_df.columns.get_loc('Systolic')
biological_end = merged_df.columns.get_loc('BMI') + 1
biological_columns = merged_df.columns[biological_start:biological_end]

R2 = []
p_values = []

for column in biological_columns:
    x = merged_df[column]
    y = merged_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/cBAG_vs_{column}.png')

cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df.to_csv('biological_metrics/BAG_vs_biological_metrics_r2_p_values.csv', index=False)
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv('biological_metrics/BAG_vs_biological_metrics_FDR_corrected_p_values.csv', index=False)


## Volume analysis for cBAG

In [ ]:
# Read the volume data
# BAG vs. Hippocampal Volume (Relative, z-scored)
# ===============================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from scipy.stats import zscore

# === Load raw regional volume data ===
vol_path = "../../normalized_regional_volumes.csv"
vol_df = pd.read_csv(vol_path)
# fill subject_id to 5 digits with leading zeros
vol_df['subject_id'] = vol_df['subject_id'].apply(lambda x: f"{int(x):05d}")
vol_df.head()

In [ ]:
# Plot the BAG vs. volume for each region with regression line, R2, and p-value
# Prepare a helper to robustly format subject IDs as zero-padded 5-digit strings
def format_id(x):
    try:
        return f"{int(float(x)):05d}"
    except Exception:
        return np.nan

# Ensure subject_id in vol_df is preserved and formatted, and convert only volume columns to numeric
if 'subject_id' in vol_df.columns:
    vol_df['subject_id'] = vol_df['subject_id'].apply(format_id)

# Choose region columns (exclude subject_id)
region_columns = [c for c in vol_df.columns if c != 'subject_id']

# Convert region columns to numeric
for col in region_columns:
    vol_df[col] = pd.to_numeric(vol_df[col], errors='coerce')

R2 = []
p_values = []

# Ensure merged_df subject_id uses the same formatting
merged_df['subject_id'] = merged_df['subject_id'].apply(format_id)

# Merge on the now-consistent string subject_id
merged_vol_df = pd.merge(vol_df, merged_df[['subject_id', 'bag_corr']], on='subject_id', how='inner')

# Set 'subject_id' as index (optional)
merged_vol_df.set_index('subject_id', inplace=True)

for column in region_columns:
    x = merged_vol_df[column]
    y = merged_vol_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/cBAG_vs_{column}.png')

# Build the results DataFrame using the same region_columns (lengths match)
volume_r2_df = pd.DataFrame({'volume_column': region_columns, 'R2': R2, 'p_value': p_values})
volume_r2_df = volume_r2_df.sort_values(by='R2', ascending=False)
top_10_volume = volume_r2_df.head(10)
print("Top 10 volume columns with highest R² values:")
print(top_10_volume)
# Print the top 10 volume columns with lowest p-values
top_10_volume_p = volume_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 volume columns with lowest p-values:")
print(top_10_volume_p)

# Compute FDR correction for p-values robustly (handle NaNs)
from statsmodels.stats.multitest import multipletests
p_arr = np.array(p_values, dtype=float)
mask = np.isfinite(p_arr)
corrected = np.full_like(p_arr, np.nan, dtype=float)
if mask.any():
    corrected_p = multipletests(p_arr[mask], method='fdr_bh')[1]
    corrected[mask] = corrected_p

# Sort by p-value and save
volume_r2_df = volume_r2_df.sort_values(by='p_value').reset_index(drop=True)
volume_r2_df['corrected_p_value'] = corrected
volume_r2_df.to_csv('biological_metrics/volume_vs_cBAG_results.csv', index=False)


## Volume analysis for BAG

In [ ]:
# Plot the BAG vs. volume for each region with regression line, R2, and p-value
# Prepare a helper to robustly format subject IDs as zero-padded 5-digit strings
def format_id(x):
    try:
        return f"{int(float(x)):05d}"
    except Exception:
        return np.nan

# Ensure subject_id in vol_df is preserved and formatted, and convert only volume columns to numeric
if 'subject_id' in vol_df.columns:
    vol_df['subject_id'] = vol_df['subject_id'].apply(format_id)

# Choose region columns (exclude subject_id)
region_columns = [c for c in vol_df.columns if c != 'subject_id']

# Convert region columns to numeric
for col in region_columns:
    vol_df[col] = pd.to_numeric(vol_df[col], errors='coerce')

R2 = []
p_values = []

# Ensure merged_df subject_id uses the same formatting
merged_df['subject_id'] = merged_df['subject_id'].apply(format_id)

# Merge on the now-consistent string subject_id
merged_vol_df = pd.merge(vol_df, merged_df[['subject_id', 'bag_raw']], on='subject_id', how='inner')

# Set 'subject_id' as index (optional)
merged_vol_df.set_index('subject_id', inplace=True)

for column in region_columns:
    x = merged_vol_df[column]
    y = merged_vol_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'biological_metrics/BAG_vs_{column}.png')

# Build the results DataFrame using the same region_columns (lengths match)
volume_r2_df = pd.DataFrame({'volume_column': region_columns, 'R2': R2, 'p_value': p_values})
volume_r2_df = volume_r2_df.sort_values(by='R2', ascending=False)
top_10_volume = volume_r2_df.head(10)
print("Top 10 volume columns with highest R² values:")
print(top_10_volume)
# Print the top 10 volume columns with lowest p-values
top_10_volume_p = volume_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 volume columns with lowest p-values:")
print(top_10_volume_p)

# Compute FDR correction for p-values robustly (handle NaNs)
from statsmodels.stats.multitest import multipletests
p_arr = np.array(p_values, dtype=float)
mask = np.isfinite(p_arr)
corrected = np.full_like(p_arr, np.nan, dtype=float)
if mask.any():
    corrected_p = multipletests(p_arr[mask], method='fdr_bh')[1]
    corrected[mask] = corrected_p

# Sort by p-value and save
volume_r2_df = volume_r2_df.sort_values(by='p_value').reset_index(drop=True)
volume_r2_df['corrected_p_value'] = corrected
volume_r2_df.to_csv('biological_metrics/volume_vs_BAG_results.csv', index=False)
